# CIS*4020 — Assignment 2 Starter Notebook
## Eager vs. Lazy Learning: Decision Trees & k-NN on OULAD

This is a **scaffold** — not the solution.

- Cells marked **PROVIDED** run as-is (data loading, the analysis table, and a few helpers).
- Cells marked **# TODO** are yours to complete.
- This notebook **is** your report: put narrative in markdown cells and code in code cells, and make it run top-to-bottom.
- Rename it **`a2_lastname.ipynb`** before submitting.

**Data:** download OULAD data and point `DATA_DIR` at the CSVs
(`https://archive.ics.uci.edu/dataset/349/open+university+learning+analytics+dataset`).

## Setup  *(PROVIDED)*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 40)
print("pandas", pd.__version__)

pandas 2.2.3


## Data loading  *(PROVIDED, same loader as Assignment 1)*

Loads the real OULAD CSVs from `DATA_DIR` if present.

In [ ]:
!wget https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip
!unzip /content/open+university+learning+analytics+dataset.zip

DATA_DIR = "/content/"   # <-- put the downloaded OULAD .csv files here for your real submission

--2026-10-28 21:34:16--  https://archive.ics.uci.edu/static/public/349/open+university+learning+analytics+dataset.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘open+university+learning+analytics+dataset.zip’

open+university+lea     [       <=>          ]  44.58M  30.6MB/s    in 1.5s    

2026-10-28 21:34:18 (30.6 MB/s) - ‘open+university+learning+analytics+dataset.zip’ saved [46748244]

Archive:  /content/open+university+learning+analytics+dataset.zip
  inflating: assessments.csv         
  inflating: courses.csv             
  inflating: studentAssessment.csv   
  inflating: studentInfo.csv         
  inflating: studentRegistration.csv  
  inflating: studentVle.csv          
  inflating: vle.csv                 
  inflating: OULAD.names             


In [ ]:
FILES = {"studentInfo":"studentInfo.csv", "studentRegistration":"studentRegistration.csv",
         "studentVle":"studentVle.csv", "vle":"vle.csv", "assessments":"assessments.csv",
         "studentAssessment":"studentAssessment.csv", "courses":"courses.csv"}

def load_oulad():
    missing = [f for f in FILES.values() if not os.path.exists(os.path.join(DATA_DIR, f))]
    if missing:
        raise FileNotFoundError(
            f"Couldn't find these OULAD files in DATA_DIR={DATA_DIR!r}: {missing}. "
            "Set DATA_DIR to the folder containing the CSVs."
        )
    print("[info] loading real OULAD from", DATA_DIR)
    return {k: pd.read_csv(os.path.join(DATA_DIR, f)) for k, f in FILES.items()}

tables = load_oulad()
for name, t in tables.items():
    print(f"{name:22s} {t.shape}")

[info] loading real OULAD from /content/
studentInfo            (32593, 12)
studentRegistration    (32593, 5)
studentVle             (10655280, 6)
vle                    (6364, 6)
assessments            (206, 6)
studentAssessment      (173912, 5)
courses                (22, 3)


In [ ]:
DATA_DIR = "."
FILES = {"studentInfo":"studentInfo.csv", "studentRegistration":"studentRegistration.csv",
         "studentVle":"studentVle.csv", "vle":"vle.csv", "assessments":"assessments.csv",
         "studentAssessment":"studentAssessment.csv", "courses":"courses.csv"}

def _make_synthetic_oulad(seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    modules, pres = ["AAA","BBB","CCC"], ["2013J","2014J"]
    imd_bands = [f"{i}-{i+10}%" for i in range(0,100,10)]
    si=[]; sr=[]; sv=[]; sid=100000
    for m in modules:
        for p in pres:
            for _ in range(250):
                eng = rng.gamma(2.0,1.0)
                imd_idx=int(rng.integers(0,10)); imd=imd_bands[imd_idx]
                if rng.random()<0.08: imd=np.nan
                dep=(9-imd_idx)/9
                pp = 1/(1+np.exp(-(-0.2 + 0.9*np.log1p(eng*14) - 1.4*dep)))
                r=rng.random()
                outcome = "Withdrawn" if (eng<0.3 and rng.random()<0.5) else (
                          "Distinction" if r<pp*0.3 else ("Pass" if r<pp else "Fail"))
                si.append({"code_module":m,"code_presentation":p,"id_student":sid,
                    "gender":rng.choice(["M","F"]),"region":rng.choice(["R1","R2","R3"]),
                    "highest_education":rng.choice(["Lower Than A Level","A Level or Equivalent","HE Qualification"]),
                    "imd_band":imd,"age_band":rng.choice(["0-35","35-55","55<="]),
                    "num_of_prev_attempts":int(rng.integers(0,3)),
                    "studied_credits":int(rng.choice([30,60,90,120])),
                    "disability":("Y" if rng.random()<0.09 else "N"),"final_result":outcome})
                sr.append({"code_module":m,"code_presentation":p,"id_student":sid,
                    "date_registration":-int(rng.integers(5,60)),
                    "date_unregistration":(int(rng.integers(0,120)) if outcome=="Withdrawn" else np.nan)})
                if rng.random()>0.05:
                    for _ in range(max(1,int(rng.poisson(eng*8)))):
                        sv.append({"code_module":m,"code_presentation":p,"id_student":sid,
                            "id_site":int(rng.integers(1,40)),"date":int(rng.integers(-10,240)),
                            "sum_click":int(1+rng.poisson(eng*3))})
                sid+=1
    return {"studentInfo":pd.DataFrame(si),"studentRegistration":pd.DataFrame(sr),"studentVle":pd.DataFrame(sv)}

def load_oulad():
    if all(os.path.exists(os.path.join(DATA_DIR,f)) for f in FILES.values()):
        print("[info] loading real OULAD from", DATA_DIR)
        return {k: pd.read_csv(os.path.join(DATA_DIR,f)) for k,f in FILES.items()}
    print("[info] OULAD files not found -> using a small SYNTHETIC stand-in so the notebook runs.")
    print("       Download the real data and set DATA_DIR before you submit.")
    return _make_synthetic_oulad()

tables = load_oulad()
for name, t in tables.items():
    print(f"{name:22s} {t.shape}")

## Build the analysis table  *(PROVIDED, mirrors Assignment 1)*

Aggregates the clickstream to one row per student-module and joins it to the student table. You may
replace this with your own Assignment 1 cleaning code.

In [ ]:
KEYS = ["code_module", "code_presentation", "id_student"]
vle = tables["studentVle"]
engagement = (vle.groupby(KEYS)
                 .agg(total_clicks=("sum_click","sum"), active_days=("date","nunique"))
                 .reset_index())
early = (vle[vle["date"] < 30].groupby(KEYS)
            .agg(early_clicks=("sum_click","sum")).reset_index())

df = (tables["studentInfo"]
      .merge(engagement, on=KEYS, how="left")
      .merge(early, on=KEYS, how="left"))
for c in ["total_clicks","active_days","early_clicks"]:
    df[c] = df[c].fillna(0)
df["imd_band"] = df["imd_band"].fillna("Missing")       # keep-and-flag (as in A1)
print("analysis table:", df.shape)
df.head()

analysis table: (32593, 15)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,total_clicks,active_days,early_clicks
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,934.0,40.0,419.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,1435.0,80.0,618.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,281.0,12.0,281.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,2158.0,123.0,503.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,1034.0,70.0,567.0


## pandas / scikit-learn quick reference

| Task | Code |
|---|---|
| stratified holdout split | `train_test_split(X, y, test_size=.2, stratify=y, random_state=42)` |
| one-hot + scale together | `ColumnTransformer([...])` |
| model = preprocess + classifier | `Pipeline([("prep", pre), ("clf", clf)])` |
| 5-fold CV score | `cross_val_score(model, X, y, cv=StratifiedKFold(5))` |
| metrics | `confusion_matrix`, `accuracy_score`, `precision_score`, `recall_score`, `f1_score` |
| decision-tree importance | `model.named_steps["clf"].feature_importances_` |
| k-NN importance | `permutation_importance(model, X_test, y_test)` |

## An evaluation helper  *(PROVIDED)*

Prints the confusion matrix and the binary metrics for the positive class.

In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score,
                             precision_score, recall_score, f1_score)
def evaluate(model, name):
    model.fit(X_tr, y_tr); pred = model.predict(X_te)
    print(f"=== {name} ===")
    print("confusion matrix [rows=actual 0/1, cols=pred 0/1]:")
    print(confusion_matrix(y_te, pred))
    print(f"accuracy  = {accuracy_score(y_te, pred):.3f}")
    print(f"precision = {precision_score(y_te, pred):.3f}   (positive class = success)")
    print(f"recall    = {recall_score(y_te, pred):.3f}")
    print(f"f1        = {f1_score(y_te, pred):.3f}")
print("helper ready")

helper ready


## Part 1 — Exploration

In [ ]:
# TODO (Part 1): explore the data and write up what you see (add markdown cells too).
#   - define your classification task and create the target variable
#   - class distribution: df["target"].value_counts()  (is it balanced?)
#   - feature types: which columns are numeric vs categorical? (NUMERIC / CATEGORICAL above)
#   - value distributions BY CLASS for a few numeric features:
#       e.g. df.boxplot(column="total_clicks", by="target")


## Part 2 — Train the models

Use `make_model(...)` so both models share the same preprocessing. Start with defaults, then explore.

In [ ]:
# TODO (Part 2): build and fit the two models.
#   from sklearn.tree import DecisionTreeClassifier
#   from sklearn.neighbors import KNeighborsClassifier
#   tree = make_model(DecisionTreeClassifier(random_state=RANDOM_SEED))
#   knn  = make_model(KNeighborsClassifier())          # defaults first
#   - then explore non-default hyperparameters:
#       tree: max_depth, min_samples_leaf       knn: n_neighbors, weights, metric
#   - for the tree, compare criterion="gini" vs criterion="entropy"


## Part 3 — Evaluation (holdout + 5-fold) and eager-vs-lazy timing

In [ ]:
# TODO (Part 3): evaluate both models and compare them.
#   - evaluate(tree, "Decision Tree");  evaluate(knn, "k-NN")
#   - 5-fold CV:  cross_val_score(make_model(...), X, y, cv=cv5, scoring="f1")  (from sklearn.model_selection)
#   - TIME them (eager vs lazy):
#       import time
#       t0=time.time(); model.fit(X_tr,y_tr); print("train", time.time()-t0)
#       t0=time.time(); model.predict(X_te); print("predict", time.time()-t0)

**TODO (markdown):** Discuss (markdown): training time, prediction time, memory. Why are they opposite?

*Your discussion here...*

## Part 4 — Feature importance

In [ ]:
# TODO (Part 4a): decision-tree feature importance.
#   fitted = make_model(DecisionTreeClassifier(random_state=RANDOM_SEED)).fit(X_tr, y_tr)
#   names = fitted.named_steps["prep"].get_feature_names_out()
#   imp   = fitted.named_steps["clf"].feature_importances_
#   pd.Series(imp, index=names).sort_values().tail(15).plot(kind="barh")
#   -> which features matter most? do they match your expectations?

# TODO (Part 4b): k-NN has no native importance -> use permutation importance.
#   from sklearn.inspection import permutation_importance
#   knn_fit = make_model(KNeighborsClassifier()).fit(X_tr, y_tr)
#   r = permutation_importance(knn_fit, X_te, y_te, n_repeats=10, random_state=RANDOM_SEED)
#   pd.Series(r.importances_mean, index=X.columns).sort_values().plot(kind="barh")
#   -> explain WHY k-NN needs this, and compare the ranking to the tree's.


## Part 5 — Eager vs. lazy learning (discussion)

**TODO (markdown):** Explain the conceptual difference between eager and lazy learners; how decision trees
and k-NN differ in training vs prediction; and the trade-offs (when would you choose each?). Back it up
with your timing numbers from Part 3.

*Your discussion here...*

## Part 6 — Reproducibility

- The random seed is set at the top (`RANDOM_SEED`).
- Before submitting: **Kernel → Restart & Run All**.

## Submission checklist
- [ ] Exploration done (class distribution, feature types, distribution plots)
- [ ] Both models trained: defaults + explored hyperparameters; tree gini vs entropy
- [ ] Confusion matrix + accuracy/precision/recall/F1 for both (+ 5-fold CV)
- [ ] Training/prediction time + memory discussed (eager vs lazy)
- [ ] Decision-tree importances + k-NN permutation importance, with discussion
- [ ] Eager vs lazy written discussion
- [ ] Runs top-to-bottom on a clean kernel; renamed `a2_lastname.ipynb`
- [ ] AI use disclosed